In [ ]:
# in the case of running in a Colab environment, we need to unzip the data first
# !unzip -q olist-sentiment-prediction-main.zip

In [ ]:
# change directory to the runs folder in colab
# %cd /content/olist-sentiment-prediction-main/Notebooks/runs

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath("../..")
sys.path.append(PROJECT_ROOT)

import time
import pandas as pd
from Scripts.corss_validation.cross_val import cross_validate
from Scripts.models.node import NODEModel

In [ ]:
# load data
df = pd.read_csv("../../Data/pre_split_features.csv")
# dataset sampling for faster execution during development
df = df.sample(n=1000,random_state=42).reset_index(drop=True)

In [ ]:
param_grid = {
    "num_layers": [2,4,8],
    "epochs": [10],
    "batch_size": [1024],
    "layer_dim": [16],
}

n_outer = 10
n_inner = 3

In [ ]:
start_time = time.time()
print("\n=== Base engineered features ===")
results = cross_validate(
    model_class=NODEModel,
    df=df,
    param_grid=param_grid,
    n_outer=n_outer,
    n_inner=n_inner,
    scoring="f1_macro",
    use_interactions=False
)
elapsed = time.time() - start_time
print(f"Execution time: {elapsed:.2f} sec")


In [ ]:
start_time = time.time()

print("\n=== Enhanced behavioural features ===")
results_enhanced = cross_validate(
    model_class=NODEModel,
    df=df,
    param_grid=param_grid,
    n_outer=n_outer,
    n_inner=n_inner,
    scoring="f1_macro",
    use_interactions=True
)

elapsed = time.time() - start_time
print(f"Execution time: {elapsed:.2f} sec")

In [ ]:
# save results
pd.DataFrame(results["fold_metrics"]).to_csv("../../Results/fold_metrics_node.csv", index=False)
pd.DataFrame(results["best_params"]).to_csv("../../Results/best_params_node.csv", index=False)

pd.DataFrame(results_enhanced["fold_metrics"]).to_csv("../../Results/fold_metrics_node_enhanced.csv", index=False)
pd.DataFrame(results_enhanced["best_params"]).to_csv("../../Results/best_params_node_enhanced.csv", index=False)

print("\nClass distribution:")
print(df["review_score"].value_counts(normalize=True))